[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SysBioChalmers/MESBcourse/blob/main/exercises/GEMextra/gemExtra.ipynb)

# GEM reconstruction in Python — *Hansenula polymorpha*

A Python / **raven-python** translation of the RAVEN MATLAB reconstruction
protocol (`reconstructionProtocol.m`) from the book chapter *"Reconstruction of
a Genome-Scale Metabolic Model for Hansenula polymorpha Using RAVEN"* (Zorrilla
& Kerkhoven). Section numbers match the MATLAB script and the book chapter.

The workflow: BLAST the *H. polymorpha* proteome against template yeasts →
build a draft model from homology → define biomass → curate lipids → gap-fill →
simulate → manual curation (methanol metabolism).

> This is the optional "explore how it works" exercise — there are no questions.
> [`raven-python`](https://github.com/SysBioChalmers/raven-python) is the Python
> port of the RAVEN toolbox; it operates on `cobra.Model` objects. A few heavy
> steps (BLAST, MILP gap-filling) and the organism-specific SLIME lipid curation
> are noted where they appear.

## 3.1 Install / setup

`raven-python` is installed from GitHub (not yet on PyPI). The homology step
needs **BLAST+** (`blastp`, `makeblastdb`); on Colab/Debian it installs with
`apt-get`. We also download the data from the public
[hanpo-GEM repository](https://github.com/SysBioChalmers/hanpo-GEM).

In [ ]:
import sys
!{sys.executable} -m pip install -q "git+https://github.com/SysBioChalmers/raven-python.git@main" cobra pandas
# BLAST+ (for the homology step). On Colab/Debian:
!apt-get -qq install -y ncbi-blast+ 2>/dev/null || echo "Install BLAST+ manually if 'blastp' is not on PATH"

In [ ]:
import os, shutil, subprocess
import pandas as pd
import cobra
print('cobra', cobra.__version__)

In [ ]:
# download the hanpo-GEM data we need
HG = "https://raw.githubusercontent.com/SysBioChalmers/hanpo-GEM/main"
files = [
    'data/templateModels/yeastGEM.xml', 'data/templateModels/rhto.xml',
    'data/genomes/hanpo.faa', 'data/genomes/sce.faa', 'data/genomes/rhto.faa',
    'data/biomass/biomassCuration.csv',
    'data/reconstruction/lipidTemplates.txt', 'data/reconstruction/lipidTransport.txt',
    'data/reconstruction/SLIMERtemplates.txt',
]
for f in files:
    os.makedirs(os.path.dirname(f), exist_ok=True)
    if not os.path.exists(f):
        !wget -q {HG}/{f} -O {f}
print('blastp available:', shutil.which('blastp') is not None)

## 3.2 Import template models

We use *S. cerevisiae* (yeast-GEM) and *R. toruloides* (rhto-GEM) as templates.
yeast-GEM is written by the COBRA Toolbox, which appends the compartment to the
metabolite ID (`s_0001[c]`); RAVEN keeps the compartment separately, so we strip
that suffix. The `id` is what `get_model_from_homology` uses to match each model
to its protein FASTA.

In [ ]:
import re
modelSce = cobra.io.read_sbml_model('data/templateModels/yeastGEM.xml')
for m in modelSce.metabolites:          # strip COBRA '[c]' compartment suffix
    m.id = re.sub(r'\[[a-z]+\]$', '', m.id)
modelSce.id = 'sce'

modelRhto = cobra.io.read_sbml_model('data/templateModels/rhto.xml')  # RAVEN-written, no suffix
modelRhto.id = 'rhto'
print(f'sce: {len(modelSce.reactions)} rxns, {len(modelSce.genes)} genes')
print(f'rhto: {len(modelRhto.reactions)} rxns, {len(modelRhto.genes)} genes')

## 3.3 Generate a draft model from homology

BLAST the whole *H. polymorpha* proteome against the *S. cerevisiae* and
*R. toruloides* proteomes (`run_blast` = RAVEN's `getBlast`; needs BLAST+, takes
a few minutes), then build a draft from the bidirectional best hits
(`get_model_from_homology` = `getModelFromHomology`). The cut-offs match the
MATLAB call (e-value 1e-30, ≥150 aa alignment, ≥35% identity).

In [ ]:
from raven_python.reconstruction.homology import run_blast, get_model_from_homology

if shutil.which('blastp'):
    blast = run_blast('hanpo', 'data/genomes/hanpo.faa',
                      ['sce', 'rhto'], ['data/genomes/sce.faa', 'data/genomes/rhto.faa'])
    blast.to_csv('blast_hits.tsv', sep='\t', index=False)   # cache for re-use
else:
    # No BLAST+? load a precomputed hits table instead (same columns as run_blast)
    blast = pd.read_csv('blast_hits.tsv', sep='\t')
print('BLAST hits:', blast.shape)

result = get_model_from_homology([modelSce, modelRhto], blast, 'hanpo',
                                 max_evalue=1e-30, min_align_len=150, min_identity=35)
model = result.model
print(f'Draft model from homology: {len(model.reactions)} rxns, {len(model.genes)} genes')

Add exchange reactions for the minimal-medium components (the MATLAB uses
`addRxnsGenesMets` to copy them from the *S. cerevisiae* template). Here we copy
the same reactions with cobrapy.

In [ ]:
medium = ['r_1654', 'r_1672', 'r_1808', 'r_1832', 'r_1861',
          'r_1992', 'r_2005', 'r_2060', 'r_2100', 'r_2111']

from cobra import Reaction, Metabolite

def copy_rxns_from(target, source, rxn_ids):
    """≈ RAVEN addRxnsGenesMets: copy reactions (with metabolites/genes) from source into target."""
    added = []
    for rid in rxn_ids:
        if rid not in source.reactions or rid in target.reactions:
            continue
        sr = source.reactions.get_by_id(rid)
        nr = Reaction(sr.id, name=sr.name, subsystem=sr.subsystem,
                      lower_bound=sr.lower_bound, upper_bound=sr.upper_bound)
        target.add_reactions([nr])
        coeffs = {}
        for met, coef in sr.metabolites.items():
            tm = (target.metabolites.get_by_id(met.id) if met.id in target.metabolites
                  else Metabolite(met.id, name=met.name, formula=met.formula,
                                  charge=met.charge, compartment=met.compartment))
            coeffs[tm] = coef
        nr.add_metabolites(coeffs)
        nr.gene_reaction_rule = sr.gene_reaction_rule
        added.append(nr.id)
    return added

print('added medium exchanges:', copy_rxns_from(model, modelSce, medium))

## 3.4 Define biomass composition

*H. polymorpha* has poly-unsaturated fatty acids like *R. toruloides*, so we
take the biomass pseudoreactions from rhto-GEM and re-define their
stoichiometry from the curated `biomassCuration.csv` (DNA, RNA, protein,
carbohydrate, lipid backbones and chains).

In [ ]:
# bring in the rhto biomass pseudoreactions + a few lipid pseudoreactions
biomass_rxns = [r.id for r in modelRhto.reactions if r.name.endswith('pseudoreaction')]
copy_rxns_from(model, modelRhto, biomass_rxns + ['r_4062', 'r_4064', 'r_4046'])
for rid in ['r_4062', 'r_4064', 'r_4046']:
    model.reactions.get_by_id(rid).bounds = (0, 1000)

BM = pd.read_csv('data/biomass/biomassCuration.csv')   # metabolite, metID, pseudoreaction, coeff
# map each biomass component group to its pseudoreaction id
groups = {'DNA': 'r_4050', 'RNA': 'r_4049', 'AA': 'r_4047',
          'carbohydrate': 'r_4048', 'backbone': 'r_4063', 'chain': 'r_4065'}

def set_biomass(rid, sub):
    rxn = model.reactions.get_by_id(rid)
    rxn.subtract_metabolites(rxn.metabolites)          # clear
    coeffs = {model.metabolites.get_by_id(met): float(c)
              for met, c in zip(sub['metID'], sub['coeff']) if met in model.metabolites}
    rxn.add_metabolites(coeffs)

for key, rid in groups.items():
    sub = BM[BM['pseudoreaction'].str.contains(key, na=False)]
    if len(sub) and rid in model.reactions:
        set_biomass(rid, sub)
print('biomass pseudoreactions updated')

## 3.5 Curation of lipid reactions (SLIME)

*H. polymorpha* has a distinctive fatty-acid / lipid-class composition. The
MATLAB protocol uses **SLIME** (Split Lipids Into Measurable Entities): each
lipid species is split into a backbone plus its acyl chains, with the chain
distribution defined in template files (`lipidTemplates.txt`,
`SLIMERtemplates.txt`). The helper functions `addLipidReactions`,
`addSLIMEreactions` and `scaleLipids` live in
[`hanpo-GEM/code/lipidMetabolism`](https://github.com/SysBioChalmers/hanpo-GEM/tree/main/code/lipidMetabolism).

This step is organism-specific and not part of raven-python; below we just load
and preview the SLIME templates to show their structure. A faithful Python port
of the three helper functions would read these tables and add the corresponding
reactions/metabolites with cobrapy — left as an extension.

In [ ]:
slime = pd.read_csv('data/reconstruction/SLIMERtemplates.txt', sep='\t')
print('SLIME templates:', slime.shape)
slime.head()

## 3.6 Gap-filling

`connect_blocked_reactions` (≈ RAVEN's `fillGaps`) finds draft reactions that
cannot carry flux and adds the minimum-penalty set of template reactions that
unblocks them — a MILP over the template models as a reaction database. (It runs
flux variability analysis first, so it is **slow** on a full model; use
`model.solver = "gurobi"` if available.) We give a generous glycerol uptake so
the chosen fluxes clear the solver tolerance.

> Note: this is the "connect blocked reactions" flavour of gap-filling. The
> MATLAB script additionally *forces* biomass production; for that specific goal
> (add reactions to make an objective feasible) use
> `cobra.flux_analysis.gapfill` after aligning the template metabolite ids.

In [ ]:
from raven_python.gapfilling import connect_blocked_reactions

model.objective = 'r_4041'                                # biomass
if 'r_1808' in model.reactions:
    model.reactions.get_by_id('r_1808').lower_bound = -10  # glycerol uptake

gap = connect_blocked_reactions(model, [modelSce, modelRhto])
print('Gap-filling: added', len(gap.added_reactions), 'reactions;',
      len(gap.newly_connected), 'newly connected,', len(gap.cannot_connect), 'still blocked')
model = gap.model

## 3.8 Simulation

Flux balance analysis — does the draft model grow?

In [ ]:
model.objective = 'r_4041'
sol = model.optimize()
print(f'Predicted growth rate: {sol.objective_value:.4f} /h ({sol.status})')
print(model.summary())

## 3.9 Manual curation — growth on methanol

*H. polymorpha* is a **methylotrophic** yeast: it grows on methanol. The
assimilation pathway needs methanol oxidase (MOX) and dihydroxyacetone synthase
(DAS), which the homology draft lacks (no close *S. cerevisiae*/*R. toruloides*
homolog). We add them manually, plus a methanol exchange, and check growth on
methanol.

In [ ]:
def met_by_name(name):
    """Resolve a metabolite by its (name, compartment) the way the MATLAB metNames[comp] does."""
    hits = [m for m in model.metabolites if m.name == name]
    return hits

# Methanol oxidase (MOX) and dihydroxyacetone synthase (DAS), in the peroxisome [p].
# These have no close sce/rhto homolog, so they are added by manual curation
# (mirroring §3.9; gene ids from H. polymorpha).
new_rxns = {
    'MOX': ('methanol oxidase', '1.1.3.13', 'Hanpo2_76277',
            {'methanol': -1, 'oxygen': -1, 'formaldehyde': 1, 'hydrogen peroxide': 1}),
    'DAS': ('dihydroxyacetone synthase', '2.2.1.3', 'Hanpo2_95557',
            {'formaldehyde': -1, 'D-xylulose 5-phosphate': -1,
             'glyceraldehyde 3-phosphate': 1, 'glycerone': 1}),
}
for rid, (name, ec, gene, stoich) in new_rxns.items():
    if rid in model.reactions:
        continue
    rxn = Reaction(rid, name=name, lower_bound=0, upper_bound=1000)
    model.add_reactions([rxn])
    coeffs = {}
    for met_name, c in stoich.items():
        cand = [m for m in met_by_name(met_name) if m.compartment in ('p', None)] or met_by_name(met_name)
        coeffs[cand[0]] = c          # uses the peroxisomal species where present
    rxn.add_metabolites(coeffs)
    rxn.gene_reaction_rule = gene
    rxn.annotation['ec-code'] = ec
print('Added methanol-assimilation reactions:', [r for r in new_rxns if r in model.reactions])
# (then: add a methanol exchange + c<->p transport, allow only methanol uptake, and re-optimise)

## Notes

- **raven-python** ([repo](https://github.com/SysBioChalmers/raven-python)) ports
  the RAVEN reconstruction stack — `run_blast`, `get_model_from_homology`,
  `connect_blocked_reactions` — onto cobrapy `Model` objects. It is alpha and
  installed from git.
- **BLAST** is an external dependency (`blastp`/`makeblastdb`). The notebook
  installs `ncbi-blast+` and caches the hits to `blast_hits.tsv`, so the homology
  step can be re-run from the cache without BLAST.
- **SLIME lipid curation** (§3.5) is organism-specific (helper functions in the
  hanpo-GEM repo), shown here as data preview; a full port reads the template
  tables and adds the reactions with cobrapy.
- **Gap-filling** is a MILP and can be slow with the default GLPK solver; switch
  with `model.solver = "gurobi"`.
- To **build a GEM for your own organism**, swap in its proteome FASTA and a
  suitable template model, and re-run from §3.3 — exactly as the MATLAB protocol
  suggests.